# Statistical Analysis — Mann-Kendall Trend Tests

This notebook detects **monotonic trends** across framework versions for the metrics
collected during the benchmark.

**Test used:** Mann-Kendall with Hamed-Rao variance correction (1998) for serial autocorrelation.  
Versions are sorted by semantic version using `packaging.version.Version`.  
For frameworks with fewer than 10 versions, the `original_test` is used as a fallback.

**Reference:**  
Hussain, M. & Mahmud, I. (2019). *pyMannKendall: a python package for non parametric Mann Kendall family of trend tests.* Journal of Open Source Software, 4(39), 1556. https://doi.org/10.21105/joss.01556


## 1. Dependencies

In [1]:
import sys
!{sys.executable} -m pip install pymannkendall packaging plotly==6.7.0


  Using cached pymannkendall-1.4.3-py3-none-any.whl.metadata (14 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
Using cached plotly-6.7.0-py3-none-any.whl (9.9 MB)
Using cached pymannkendall-1.4.3-py3-none-any.whl (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pymannkendall]


## 2. Imports and configuration

In [1]:
from pathlib import Path
import pandas as pd

from mann_kendall_analysis import (
    run_mk_analysis,
    mk_summary_table,
    mk_pivot_table,
    plot_mk_heatmap,
    plot_mk_trend_line,
    export_mk_results,
    DEFAULT_METRICS,
)

# ── Configuration ─────────────────────────────────────────────────────────────
SUMMARY_CSV = Path("summary_by_version.csv")  # produced by aggregate_results.ipynb
ALPHA       = 0.05
VARIANT     = "hamed_rao"  # "hamed_rao" | "original" | "yue_wang"
print("Done.")


## 3. Load data

In [2]:
df_summary = pd.read_csv(SUMMARY_CSV)

print(f"Frameworks: {sorted(df_summary['framework'].unique())}")
print(f"\nVersions per framework:")
print(df_summary.groupby('framework')['framework_version'].count().sort_values())


Frameworks: ['aiohttp', 'baize', 'django', 'fastapi', 'muffin', 'starlette']

Versions per framework:
framework
baize         30
muffin        48
aiohttp       61
django       113
starlette    130
fastapi      191
Name: framework_version, dtype: int64


# 3b. (Optional) Load pre-computed results

If data exists from previous run, can load directly and skip the simulation.

In [9]:
MK_RESULTS_CSV = Path("mk_results.csv")

if MK_RESULTS_CSV.exists():
    mk_results = pd.read_csv(MK_RESULTS_CSV)
    # Restore boolean column — CSV serialises it as 0/1
    mk_results["h"] = mk_results["h"].astype(bool)
    print(f"Loaded {len(mk_results)} rows from {MK_RESULTS_CSV}")
    print(f"Trend distribution: {mk_results['trend'].value_counts()}")
    print(f"Significant results (h=True): {mk_results['h'].sum()}")
else:
    print(f"{MK_RESULTS_CSV} not found — run Section 4 to compute results.")
    mk_results = None


Loaded 66 rows from mk_results.csv
Trend distribution: trend
increasing    24
decreasing    22
no trend      20
Name: count, dtype: int64
Significant results (h=True): 46


## 4. Run Mann-Kendall analysis

Runs the test for every (framework, metric) combination.  
Frameworks with fewer than 4 versions are skipped automatically.


In [3]:
mk_results = run_mk_analysis(
    df_summary,
    alpha=ALPHA,
    variant=VARIANT,
)

print(f"Total analyses: {len(mk_results)}")
print(f"\nTrend distribution:")
print(mk_results["trend"].value_counts())
print(f"\nStatistically significant results (h=True): {mk_results['h'].sum()}")


Total analyses: 84

Trend distribution:
trend
increasing    33
no trend      26
decreasing    25
Name: count, dtype: int64

Statistically significant results (h=True): 58


## 5. Results table — significant trends

In [4]:
sig_table = mk_summary_table(mk_results, significant_only=True, sort_by="framework")
display(sig_table)


,framework,metric_label,group,n_versions,trend,h,p_value,tau,slope,note
0,aiohttp,Total energy (kWh),energy,61,decreasing,True,0.0018,-0.539,-5.003e-08,
1,aiohttp,HTML latency p95 (s),latency,61,decreasing,True,0.0000,-0.722,-4.284e-05,
2,aiohttp,HTML latency Mean(s),latency,61,decreasing,True,0.0000,-0.718,-3.318e-05,
3,aiohttp,API latency Mean(s),latency,61,decreasing,True,0.0014,-0.634,-9.449e-05,
4,aiohttp,HTML throughput (req/s),throughput,61,increasing,True,0.0000,+0.718,4.294e+01,
5,aiohttp,API latency p95 (s),latency,61,decreasing,True,0.0037,-0.558,-9.854e-05,
6,aiohttp,CO2 emitted (kg),carbon,61,decreasing,True,0.0018,-0.539,-3.327e-08,
7,aiohttp,RAM energy (kWh),energy,61,decreasing,True,0.0011,-0.501,-3.177e-08,
8,aiohttp,CPU energy (kWh),energy,61,decreasing,True,0.0045,-0.532,-1.520e-08,
9,aiohttp,API throughput (req/s),throughput,61,increasing,True,0.0021,+0.642,7.935e+01,


## 6. Results by metric group

In [5]:
for group in ["energy", "carbon", "throughput", "latency"]:
    print(f"\n{'='*60}")
    print(f"  Group: {group.upper()}")
    print('='*60)
    tbl = mk_summary_table(mk_results, group=group, sort_by="framework")
    display(tbl)



  Group: ENERGY


,framework,metric_label,group,n_versions,trend,h,p_value,tau,slope,note
0,aiohttp,Total energy (kWh),energy,61,decreasing,True,0.0018,-0.539,-5.003e-08,
1,aiohttp,CPU energy (kWh),energy,61,decreasing,True,0.0045,-0.532,-1.520e-08,
2,aiohttp,RAM energy (kWh),energy,61,decreasing,True,0.0011,-0.501,-3.177e-08,
3,baize,Total energy (kWh),energy,30,no trend,False,0.2520,+0.205,2.552e-08,
4,baize,CPU energy (kWh),energy,30,no trend,False,0.3524,+0.186,8.112e-09,
5,baize,RAM energy (kWh),energy,30,no trend,False,0.3023,+0.186,1.741e-08,
6,django,CPU energy (kWh),energy,113,increasing,True,0.0011,+0.552,2.018e-08,
7,django,RAM energy (kWh),energy,113,increasing,True,0.0001,+0.578,1.097e-07,
8,django,Total energy (kWh),energy,113,increasing,True,0.0002,+0.578,1.281e-07,
9,fastapi,Total energy (kWh),energy,191,increasing,True,0.0157,+0.324,1.202e-08,



  Group: CARBON


,framework,metric_label,group,n_versions,trend,h,p_value,tau,slope,note
0,aiohttp,CO2 emitted (kg),carbon,61,decreasing,True,0.0018,-0.539,-3.327e-08,
1,aiohttp,CO2 rate (kg CO2/s),carbon,61,no trend,False,0.4005,-0.086,-1.332e-10,
2,baize,CO2 emitted (kg),carbon,30,no trend,False,0.2520,+0.205,1.697e-08,
3,baize,CO2 rate (kg CO2/s),carbon,30,no trend,False,0.1535,-0.186,-4.020e-10,
4,django,CO2 emitted (kg),carbon,113,increasing,True,0.0002,+0.578,8.518e-08,
5,django,CO2 rate (kg CO2/s),carbon,113,decreasing,True,0.0000,-0.546,-3.380e-10,
6,fastapi,CO2 emitted (kg),carbon,191,increasing,True,0.0157,+0.324,7.995e-09,
7,fastapi,CO2 rate (kg CO2/s),carbon,191,no trend,False,0.1854,+0.103,5.213e-11,
8,muffin,CO2 emitted (kg),carbon,48,increasing,True,0.0224,+0.229,5.487e-09,
9,muffin,CO2 rate (kg CO2/s),carbon,48,no trend,False,0.4716,+0.073,2.065e-10,



  Group: THROUGHPUT


,framework,metric_label,group,n_versions,trend,h,p_value,tau,slope,note
0,aiohttp,API throughput (req/s),throughput,61,increasing,True,0.0021,+0.642,7.935e+01,
1,aiohttp,HTML throughput (req/s),throughput,61,increasing,True,0.0000,+0.718,4.294e+01,
2,aiohttp,Upload throughput (req/s),throughput,61,no trend,False,0.9575,-0.009,-7.005e-02,
3,baize,API throughput (req/s),throughput,30,no trend,False,0.8028,-0.034,-1.522e+00,
4,baize,HTML throughput (req/s),throughput,30,no trend,False,0.3702,-0.108,-8.049e+00,
5,baize,Upload throughput (req/s),throughput,30,decreasing,True,0.0489,-0.421,-5.389e+00,
6,django,HTML throughput (req/s),throughput,113,decreasing,True,0.0000,-0.648,-4.856e+00,
7,django,Upload throughput (req/s),throughput,113,decreasing,True,0.0006,-0.555,-1.753e+00,
8,django,API throughput (req/s),throughput,113,decreasing,True,0.0000,-0.639,-4.176e+00,
9,fastapi,API throughput (req/s),throughput,191,decreasing,True,0.0001,-0.671,-1.878e+01,



  Group: LATENCY


,framework,metric_label,group,n_versions,trend,h,p_value,tau,slope,note
0,aiohttp,API latency Mean(s),latency,61,decreasing,True,0.0014,-0.634,-9.449e-05,
1,aiohttp,HTML latency Mean(s),latency,61,decreasing,True,0.0000,-0.718,-3.318e-05,
2,aiohttp,Upload latency Mean(s),latency,61,no trend,False,0.5530,-0.091,-1.125e-05,
3,aiohttp,API latency p95 (s),latency,61,decreasing,True,0.0037,-0.558,-9.854e-05,
4,aiohttp,HTML latency p95 (s),latency,61,decreasing,True,0.0000,-0.722,-4.284e-05,
5,aiohttp,Upload latency p95 (s),latency,61,no trend,False,0.4538,-0.144,-2.771e-05,
6,baize,API latency Mean(s),latency,30,no trend,False,0.3973,+0.140,1.355e-05,
7,baize,HTML latency Mean(s),latency,30,no trend,False,0.7486,+0.067,3.128e-06,
8,baize,Upload latency Mean(s),latency,30,no trend,False,0.1898,+0.301,7.575e-05,
9,baize,API latency p95 (s),latency,30,no trend,False,0.3582,+0.149,2.350e-05,


## 7. Pivot tables — framework × metric

Suitable for inclusion in the paper as summary tables.


In [6]:
print("=== Trend direction — Energy ===")
display(mk_pivot_table(mk_results, value="trend", group="energy"))

print("\n=== p-values — Energy ===")
display(mk_pivot_table(mk_results, value="p_value", group="energy"))

print("\n=== Kendall's Tau — Throughput ===")
display(mk_pivot_table(mk_results, value="tau", group="throughput"))

print("\n=== Kendall's Tau — Latency ===")
display(mk_pivot_table(mk_results, value="tau", group="latency"))


=== Trend direction — Energy ===


metric_label,CPU energy (kWh),RAM energy (kWh),Total energy (kWh)
framework,,,
aiohttp,decreasing,decreasing,decreasing
baize,no trend,no trend,no trend
django,increasing,increasing,increasing
fastapi,increasing,increasing,increasing
muffin,no trend,increasing,increasing
starlette,decreasing,decreasing,decreasing



=== p-values — Energy ===


metric_label,CPU energy (kWh),RAM energy (kWh),Total energy (kWh)
framework,,,
aiohttp,0.0045,0.0011,0.0018
baize,0.3524,0.3023,0.2520
django,0.0011,0.0001,0.0002
fastapi,0.0268,0.0087,0.0157
muffin,0.3580,0.0322,0.0224
starlette,0.0030,0.0038,0.0062



=== Kendall's Tau — Throughput ===


metric_label,API throughput (req/s),HTML throughput (req/s),Upload throughput (req/s)
framework,,,
aiohttp,+0.642,+0.718,-0.009
baize,-0.034,-0.108,-0.421
django,-0.639,-0.648,-0.555
fastapi,-0.671,-0.611,-0.344
muffin,+0.028,-0.387,-0.388
starlette,-0.401,-0.238,+0.441



=== Kendall's Tau — Latency ===


metric_label,API latency Mean(s),API latency p95 (s),HTML latency Mean(s),HTML latency p95 (s),Upload latency Mean(s),Upload latency p95 (s)
framework,,,,,,
aiohttp,-0.634,-0.558,-0.718,-0.722,-0.091,-0.144
baize,+0.140,+0.149,+0.067,+0.246,+0.301,+0.292
django,+0.626,+0.671,+0.615,+0.668,+0.556,+0.649
fastapi,+0.667,+0.629,+0.580,+0.544,+0.347,+0.311
muffin,+0.067,+0.116,+0.438,+0.328,+0.415,+0.390
starlette,+0.393,+0.166,+0.234,+0.269,-0.451,-0.514


## 8. Trend heatmaps

Color key:
- 🔴 **Red** — statistically significant *increasing* trend (worsening for energy/latency; improvement for throughput)
- 🟢 **Green** — statistically significant *decreasing* trend
- 🔵 **Light blue** — no statistically significant trend
- ⬜ **Grey** — insufficient data / skipped

> **Interpretation note:** "increasing" is unfavourable for energy and latency metrics,
> but favourable for throughput. The heatmap colour always encodes the raw direction —
> domain interpretation is the reader's responsibility.


In [7]:
# Heatmap — Energy & Carbon metrics
fig_energy_carbon = plot_mk_heatmap(
    mk_results,
    metric_group=["energy", "carbon"],
    alpha=ALPHA,
    title=f"Mann-Kendall — Energy & Carbon metrics (α={ALPHA})",
)
fig_energy_carbon.show(renderer="iframe_connected")


In [8]:
# Heatmap — Throughput metrics (higher is better: increasing = green)
fig_throughput = plot_mk_heatmap(
    mk_results,
    metric_group="throughput",
    alpha=ALPHA,
    title=f"Mann-Kendall — Throughput metrics (α={ALPHA})",
    higher_is_better=True,
)
fig_throughput.show(renderer="iframe_connected")

In [9]:
# Heatmaps per group — export to HTML
# Energy and carbon are exported together; throughput uses higher_is_better
fig_energy_carbon.write_html("./plots/cmk_heatmap_energy_carbon.html", include_plotlyjs="cdn")

for group in ["throughput", "latency"]:
    print(group)
    higher_is_better = (group == "throughput")
    fig = plot_mk_heatmap(
        mk_results,
        metric_group=group,
        alpha=ALPHA,
        higher_is_better=higher_is_better,
    )
    fig.write_html(f"./plots/cmk_heatmap_{group}.html", include_plotlyjs="cdn")


throughput
latency


## 9. Drill-down: time series with Theil-Sen trend line

Each plot shows the metric value across versions (mean ± std band) with the
Theil-Sen regression line overlaid. The slope unit is *metric unit per version step*.


In [13]:
# Single framework / metric
FRAMEWORK = "fastapi"
METRIC    = "emission_emissions_mean"

mk_row = mk_results[
    (mk_results["framework"] == FRAMEWORK) &
    (mk_results["metric"]    == METRIC)
]

fig = plot_mk_trend_line(
    df_summary,
    framework=FRAMEWORK,
    metric=METRIC,
    mk_result_row=mk_row.iloc[0] if len(mk_row) > 0 else None,
)
fig.show(renderer="iframe_connected")


In [15]:
# Loop: Theil-Sen trend line for energy_consumed across all frameworks
for fw in sorted(df_summary["framework"].unique()):
    mk_row = mk_results[
        (mk_results["framework"] == fw) &
        (mk_results["metric"]    == "emission_emissions_mean")
    ]
    if mk_row.empty:
        continue
    fig = plot_mk_trend_line(
        df_summary,
        framework=fw,
        metric="emission_emissions_mean",
        mk_result_row=mk_row.iloc[0],
    )
    fig.write_html(f"./plots/{fw}/mk_theil_sen_trend_{fw}.html", include_plotlyjs="cdn")
print("Done.")


Done.


## 10. Export results

In [12]:
export_mk_results(
    mk_results,
    output_csv="mk_results.csv",
    output_html="./plots/mk_heatmap.html",
    alpha=ALPHA,
)

# Export combined energy+carbon and throughput heatmaps
fig_energy_carbon.write_html("./plots/mk_heatmap_energy_carbon.html", include_plotlyjs="cdn")
fig_throughput.write_html("./plots/mk_heatmap_throughput.html", include_plotlyjs="cdn")

print("\nGenerated files:")
print("  mk_results.csv                      — full results table (CSV)")
print("  mk_heatmap_energy_carbon.html   — energy & carbon trend heatmap")
print("  mk_heatmap_throughput.html      — throughput trend heatmap")


Results saved to: mk_results.csv
Heatmap saved to: ./plots/mk_heatmap.html

Generated files:
  mk_results.csv                      — full results table (CSV)
  mk_heatmap_energy_carbon.html   — energy & carbon trend heatmap
  mk_heatmap_throughput.html      — throughput trend heatmap
